# Heterogeneous Population Space-Time Ve Generator

For every manually exported COMSOL cutline and requested MRG diameter, this notebook first creates a 1 µA spatial Ve basis. It then applies every shared population time-pattern bundle to each basis and saves one `SpaceTimeV` file per `(axon_id, CV, frequency)`.

Files use the single-axon-compatible fields `time_s`, `node_numbers`, `rep_labels`, `current`, and `V`, with additive axon metadata. Example: `S01_57um_1uA_cv0.0_f10_SpaceTimeV.npz`.

Every whole MRG internodal step that fits the manual COMSOL cutline is retained. No Ve values are extrapolated or padded with a terminal sample: smaller fibers can therefore have more than the legacy 26 nodes, while the last retained node is recorded as the valid readout node.

The complete 35-trajectory × 6-diameter cohort contains 11,550 SpaceTimeV archives. For the production run, use `slurm/run_population_pipeline.sh` to generate these in parallel on Longleaf; this notebook remains the equivalent local/resumable path.

In [ ]:
# ---------------- User inputs ----------------

from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve().parents[1]
SOURCE_DIR = REPO_ROOT / 'src'
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

DIAMETERS_UM = (5.7, 8.7, 10.0, 11.5, 12.8, 14.0)
CV_VALS = tuple(round(0.1 * i, 1) for i in range(11))
FREQ_VALS = (10, 20, 50, 100, 200)

COMSOL_DIR = REPO_ROOT / 'data' / 'comsol'
TIME_PATTERN_DIR = REPO_ROOT / 'data' / 'time-patterns' / 'population'
OUT_DIR = REPO_ROOT / 'data' / 'space-time-V'


In [ ]:
# ---------------- Generation helpers ----------------

from space_time_v.spacetime_core import (
    discover_comsol_cutline_ids,
    generate_population_spacetime,
    generate_population_ve_bases,
)

CUTLINE_IDS = discover_comsol_cutline_ids(COMSOL_DIR)
print(f'Discovered {len(CUTLINE_IDS)} complete COMSOL cutline pairs.')


In [ ]:
# ---------------- Build all axon bases ----------------

basis_rows = generate_population_ve_bases(
    CUTLINE_IDS,
    DIAMETERS_UM,
    comsol_dir=COMSOL_DIR,
    out_dir=OUT_DIR,
)
print(f'Generated {len(basis_rows)} axon-specific Ve basis files.')


In [ ]:
# ---------------- Build all axon × CV × frequency SpaceTimeV files ----------------

spacetime_rows = generate_population_spacetime(
    basis_rows,
    CV_VALS,
    FREQ_VALS,
    time_pattern_dir=TIME_PATTERN_DIR,
    out_dir=OUT_DIR,
)
print(f'Generated {len(spacetime_rows)} population SpaceTimeV files.')


In [ ]:
# ---------------- Inspect one compatible output ----------------

import numpy as np

example_path = Path(spacetime_rows[0]['spacetime_npz'])
with np.load(example_path) as data:
    print('axon_id:', data['axon_id'].item())
    print('current:', data['current'].shape)
    print('V:', data['V'].shape)
    print('valid nodes:', int(data['n_valid_nodes']))
    print('readout node:', int(data['readout_node_number']))
